# Supplementary Practical – PyTorch Fundamentals with ANN, CNN, and RNN

**Course:** ML6402 – Machine Learning  
**Instructor:** Dr. Ahmed Sabbah  
**University:** Al-Quds Open University  

This notebook is **additional knowledge**. The main deep-learning implementation in ML6402 uses TensorFlow/Keras, while this notebook introduces PyTorch for students who want a more explicit, research-oriented deep-learning workflow.

## Learning objectives

By the end of this notebook, you should be able to:

- create and manipulate PyTorch tensors;
- move tensors and models between CPU and GPU;
- use `TensorDataset` and `DataLoader`;
- define a neural network using `nn.Module`;
- understand the forward pass, loss function, backpropagation, and optimizer;
- write reusable training and evaluation loops;
- implement a simple ANN;
- implement a simple CNN;
- implement a simple RNN;
- make predictions;
- save and reload model parameters.

The examples are intentionally small so that the focus remains on the **PyTorch programming model** rather than on obtaining state-of-the-art performance.

## 1. Import PyTorch and check the execution device

PyTorch can run on the CPU or, when available, on a GPU. We choose the device once and then move both the model and its tensors to that device.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

print("PyTorch version:", torch.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. PyTorch tensors

A PyTorch tensor is similar to a NumPy array, but it can also:

- run on a GPU;
- participate in automatic differentiation;
- be used directly by neural-network layers.

In [ ]:
x = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])

print("Tensor:")
print(x)
print("Shape:", x.shape)
print("Data type:", x.dtype)

print("\nBasic operations:")
print("x + 2 =")
print(x + 2)

print("\nMatrix multiplication:")
print(x @ x.T)

### Automatic differentiation

When `requires_grad=True`, PyTorch records operations so that gradients can be computed automatically during backpropagation.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
x_value = torch.tensor(3.0)

y = w * x_value ** 2
y.backward()

print("y =", y.item())
print("dy/dw =", w.grad.item())

## 3. Dataset and DataLoader

A common PyTorch workflow is:

`NumPy/Pandas → Tensor → Dataset → DataLoader`

A `DataLoader` divides the data into mini-batches and can shuffle the training samples.

In [ ]:
X_demo = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y_demo = torch.tensor([[2.0], [4.0], [6.0], [8.0]])

demo_dataset = TensorDataset(X_demo, y_demo)
demo_loader = DataLoader(demo_dataset, batch_size=2, shuffle=True)

for X_batch, y_batch in demo_loader:
    print("X batch:", X_batch.squeeze().tolist())
    print("y batch:", y_batch.squeeze().tolist())
    print("---")

## 4. A reusable PyTorch training loop

The central training steps are:

1. obtain a mini-batch;
2. move it to the selected device;
3. clear old gradients;
4. perform the forward pass;
5. compute the loss;
6. call `loss.backward()` to compute gradients;
7. call `optimizer.step()` to update the parameters.

In [ ]:
def train_classifier(model, loader, loss_fn, optimizer, epochs=20):
    model.to(device)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * X_batch.size(0)

        if epoch == 0 or (epoch + 1) % 5 == 0:
            epoch_loss = running_loss / len(loader.dataset)
            print(f"Epoch {epoch + 1:>2}/{epochs} - loss: {epoch_loss:.4f}")


def evaluate_classifier(model, loader):
    model.eval()
    predictions = []
    targets = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)
            pred = torch.argmax(logits, dim=1)

            predictions.extend(pred.cpu().numpy())
            targets.extend(y_batch.numpy())

    return accuracy_score(targets, predictions)

# Part A – Artificial Neural Network (ANN)

We will classify the Scikit-Learn breast-cancer dataset.

The ANN demonstrates:

- fully connected `Linear` layers;
- ReLU activation;
- mini-batch training;
- `CrossEntropyLoss`;
- Adam optimizer.

In [ ]:
data = load_breast_cancer()
X = data.data.astype(np.float32)
y = data.target.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

train_dataset = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_train)
)
test_dataset = TensorDataset(
    torch.from_numpy(X_test),
    torch.from_numpy(y_test)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("Number of input features:", X_train.shape[1])
print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

## 5. Define the ANN using `nn.Module`

Every custom PyTorch network normally inherits from `nn.Module`.

- `__init__()` defines the layers.
- `forward()` defines how data moves through those layers.

In [ ]:
class ANNClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )

    def forward(self, x):
        return self.network(x)


ann_model = ANNClassifier(input_dim=X_train.shape[1])
print(ann_model)

### Why does the final layer have 2 outputs?

There are two classes. The network produces two raw scores called **logits**.

When we use `CrossEntropyLoss`, we should **not** manually apply softmax inside the model. The loss function handles the required transformation internally.

In [ ]:
ann_loss = nn.CrossEntropyLoss()
ann_optimizer = torch.optim.Adam(ann_model.parameters(), lr=0.001)

train_classifier(
    ann_model,
    train_loader,
    ann_loss,
    ann_optimizer,
    epochs=20
)

ann_accuracy = evaluate_classifier(ann_model, test_loader)
print("ANN test accuracy:", round(ann_accuracy, 4))

## 6. Make a prediction with the ANN

In [ ]:
ann_model.eval()

sample = torch.from_numpy(X_test[:1]).to(device)

with torch.no_grad():
    logits = ann_model(sample)
    predicted_class = torch.argmax(logits, dim=1).item()

print("Predicted class:", data.target_names[predicted_class])
print("Actual class   :", data.target_names[y_test[0]])

# Part B – Convolutional Neural Network (CNN)

A CNN is designed to exploit spatial structure in images.

We use the Scikit-Learn handwritten-digits dataset:

- 8 × 8 grayscale images;
- 10 classes: digits 0–9.

The example demonstrates:

- adding an image-channel dimension;
- `Conv2d`;
- ReLU;
- max pooling;
- flattening;
- fully connected classification.

In [ ]:
digits = load_digits()

X_img = digits.images.astype(np.float32) / 16.0
y_img = digits.target.astype(np.int64)

X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(
    X_img, y_img,
    test_size=0.20,
    random_state=42,
    stratify=y_img
)

# CNN input shape: [samples, channels, height, width]
X_train_img = X_train_img[:, np.newaxis, :, :]
X_test_img = X_test_img[:, np.newaxis, :, :]

train_img_dataset = TensorDataset(
    torch.from_numpy(X_train_img),
    torch.from_numpy(y_train_img)
)
test_img_dataset = TensorDataset(
    torch.from_numpy(X_test_img),
    torch.from_numpy(y_test_img)
)

train_img_loader = DataLoader(train_img_dataset, batch_size=64, shuffle=True)
test_img_loader = DataLoader(test_img_dataset, batch_size=128, shuffle=False)

print("CNN training tensor shape:", X_train_img.shape)

## 7. Define the CNN

For an 8 × 8 image:

- first convolution preserves the spatial size;
- max pooling reduces 8 × 8 to 4 × 4;
- second convolution preserves 4 × 4;
- second max pooling reduces 4 × 4 to 2 × 2.

With 32 output channels, the flattened representation therefore contains:

`32 × 2 × 2 = 128` values.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 2 * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


cnn_model = SimpleCNN()
print(cnn_model)

In [ ]:
cnn_loss = nn.CrossEntropyLoss()
cnn_optimizer = torch.optim.Adam(cnn_model.parameters(), lr=0.001)

train_classifier(
    cnn_model,
    train_img_loader,
    cnn_loss,
    cnn_optimizer,
    epochs=10
)

cnn_accuracy = evaluate_classifier(cnn_model, test_img_loader)
print("CNN test accuracy:", round(cnn_accuracy, 4))

## 8. Inspect CNN predictions

In [ ]:
cnn_model.eval()

images = torch.from_numpy(X_test_img[:8]).to(device)

with torch.no_grad():
    predictions = torch.argmax(cnn_model(images), dim=1).cpu().numpy()

fig, axes = plt.subplots(2, 4, figsize=(8, 4))

for ax, image, pred, actual in zip(
    axes.ravel(),
    X_test_img[:8, 0],
    predictions,
    y_test_img[:8]
):
    ax.imshow(image, cmap="gray")
    ax.set_title(f"P:{pred} / A:{actual}")
    ax.axis("off")

plt.tight_layout()
plt.show()

# Part C – Recurrent Neural Network (RNN)

RNNs process sequences rather than independent feature vectors.

To keep the example self-contained, we create a noisy sine-wave time series and ask the RNN to predict the next value from a fixed-length window of previous values.

This demonstrates:

- sequential input shape;
- `nn.RNN`;
- hidden states;
- selecting the final time step;
- regression with `MSELoss`.

In [ ]:
np.random.seed(42)

t = np.linspace(0, 50, 1200)
series = np.sin(t) + 0.10 * np.random.randn(len(t))

plt.plot(t[:300], series[:300])
plt.xlabel("Time")
plt.ylabel("Value")
plt.title("Synthetic time series")
plt.show()

## 9. Convert the time series into supervised sequences

If the sequence length is 20, each training example contains 20 consecutive values, and the target is the next value.

In [ ]:
SEQUENCE_LENGTH = 20

def make_sequences(values, sequence_length):
    X_seq = []
    y_seq = []

    for i in range(len(values) - sequence_length):
        X_seq.append(values[i:i + sequence_length])
        y_seq.append(values[i + sequence_length])

    X_seq = np.array(X_seq, dtype=np.float32)[..., np.newaxis]
    y_seq = np.array(y_seq, dtype=np.float32).reshape(-1, 1)

    return X_seq, y_seq


X_seq, y_seq = make_sequences(series, SEQUENCE_LENGTH)

split = int(0.80 * len(X_seq))

X_train_seq = X_seq[:split]
X_test_seq = X_seq[split:]
y_train_seq = y_seq[:split]
y_test_seq = y_seq[split:]

train_seq_dataset = TensorDataset(
    torch.from_numpy(X_train_seq),
    torch.from_numpy(y_train_seq)
)
test_seq_dataset = TensorDataset(
    torch.from_numpy(X_test_seq),
    torch.from_numpy(y_test_seq)
)

train_seq_loader = DataLoader(train_seq_dataset, batch_size=32, shuffle=True)
test_seq_loader = DataLoader(test_seq_dataset, batch_size=64, shuffle=False)

print("RNN input shape:", X_train_seq.shape)
print("[samples, time steps, features]")

## 10. Define the RNN

With `batch_first=True`, the RNN expects:

`[batch, time steps, features]`

The RNN returns outputs for every time step. For next-value prediction, we use the output from the **last time step**.

In [ ]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=32):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, x):
        rnn_output, hidden_state = self.rnn(x)

        last_time_step = rnn_output[:, -1, :]
        prediction = self.output(last_time_step)

        return prediction


rnn_model = SimpleRNN()
print(rnn_model)

## 11. Regression training loop for the RNN

The structure is almost identical to classification training. The main difference is the loss function and the form of the network output.

In [ ]:
def train_regressor(model, loader, loss_fn, optimizer, epochs=20):
    model.to(device)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            prediction = model(X_batch)
            loss = loss_fn(prediction, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * X_batch.size(0)

        if epoch == 0 or (epoch + 1) % 5 == 0:
            epoch_loss = running_loss / len(loader.dataset)
            print(f"Epoch {epoch + 1:>2}/{epochs} - MSE: {epoch_loss:.5f}")


rnn_loss = nn.MSELoss()
rnn_optimizer = torch.optim.Adam(rnn_model.parameters(), lr=0.001)

train_regressor(
    rnn_model,
    train_seq_loader,
    rnn_loss,
    rnn_optimizer,
    epochs=20
)

## 12. Evaluate the RNN

In [ ]:
rnn_model.eval()

rnn_predictions = []

with torch.no_grad():
    for X_batch, _ in test_seq_loader:
        X_batch = X_batch.to(device)
        pred = rnn_model(X_batch)
        rnn_predictions.extend(pred.cpu().numpy().ravel())

rnn_predictions = np.array(rnn_predictions)

mse = np.mean((rnn_predictions - y_test_seq.ravel()) ** 2)
print("RNN test MSE:", round(float(mse), 5))

plt.figure(figsize=(10, 4))
plt.plot(y_test_seq[:150], label="Actual")
plt.plot(rnn_predictions[:150], label="Predicted")
plt.xlabel("Test sequence")
plt.ylabel("Value")
plt.legend()
plt.show()

## RNN versus LSTM

`nn.RNN` is useful for understanding the recurrent idea, but plain RNNs can have difficulty learning long-term dependencies.

PyTorch also provides:

- `nn.LSTM`
- `nn.GRU`

For example, replacing:

```python
self.rnn = nn.RNN(...)
```

with:

```python
self.rnn = nn.LSTM(...)
```

creates an LSTM-based recurrent layer. The overall training workflow remains very similar.

## 13. Save and reload model parameters

The recommended PyTorch pattern is to save the model's **state dictionary**, not the whole Python object.

In [ ]:
# Save ANN parameters
torch.save(ann_model.state_dict(), "ann_model_weights.pth")

# Create a new model with the same architecture
reloaded_ann = ANNClassifier(input_dim=X_train.shape[1])

# Load parameters
reloaded_ann.load_state_dict(
    torch.load("ann_model_weights.pth", map_location=device)
)
reloaded_ann.to(device)
reloaded_ann.eval()

print("Model weights reloaded successfully.")

## 14. PyTorch workflow summary

The typical PyTorch workflow is:

1. **Prepare data**
   - NumPy / Pandas
   - convert to tensors
   - create Dataset
   - create DataLoader

2. **Define model**
   - subclass `nn.Module`
   - define layers in `__init__()`
   - define computation in `forward()`

3. **Choose training components**
   - loss function
   - optimizer
   - learning rate

4. **Train**
   - `model.train()`
   - zero gradients
   - forward pass
   - compute loss
   - `loss.backward()`
   - `optimizer.step()`

5. **Evaluate**
   - `model.eval()`
   - `torch.no_grad()`
   - calculate predictions and metrics

6. **Save**
   - `torch.save(model.state_dict(), ...)`

These steps are common across ANN, CNN, RNN, LSTM, Transformers, and many other PyTorch models.

## 15. Student exercises

### ANN
1. Change the hidden layers from `32 → 16` to `64 → 32 → 16`.
2. Compare Adam with SGD.
3. Try different learning rates.

### CNN
1. Remove one convolutional layer and compare accuracy.
2. Change the number of filters.
3. Explain why the input shape contains a channel dimension.

### RNN
1. Change the sequence length from 20 to 10 and 40.
2. Replace `nn.RNN` with `nn.LSTM`.
3. Compare the prediction curve and MSE.

### General PyTorch
1. Identify where the forward pass occurs.
2. Identify where gradients are calculated.
3. Explain the roles of `model.train()` and `model.eval()`.
4. Explain why `optimizer.zero_grad()` is called before backpropagation.
5. Explain why evaluation code uses `torch.no_grad()`.